# Import

In [ ]:
import numpy as np
import healpy as hp
import pandas as pd
import networkx as nx
import geopandas as gpd

from itertools import combinations

from pyproj import Geod
from shapely import union_all, intersects
from shapely.wkt import loads
from shapely.geometry import LineString
from healpix_geo.nested import healpix_to_lonlat
from healpix_geo.nested import lonlat_to_healpix

import plotly.express as px
import plotly.graph_objects as go

import matplotlib.pyplot as plt

In [ ]:
def get_points(NSIDE):
    NPIX = hp.nside2npix(NSIDE)
    points = np.array(
        hp.pix2vec(NSIDE, range(NPIX), nest=True)
    ).T

    df = pd.DataFrame(points, columns=('x', 'y', 'z'))

    return df

def get_patches(NSIDE):
    NPIX = hp.nside2npix(NSIDE)

    pixels = np.arange(NPIX)
    patches = np.array(
        [hp.boundaries(NSIDE, i, nest=True).T for i in pixels]
    ).reshape(-1, 3)

    df = pd.DataFrame(patches, columns=('x', 'y', 'z'))

    return df

In [ ]:
def great_circle_path(a=None, b=None, n_points=100, iterable=False):
    geod = Geod(ellps='WGS84')

    if iterable:
        (a, b) = a

    points = geod.npts(
        a.x, a.y,
        b.x, b.y,
        n_points,
    )
    line = LineString(points)

    return line

def great_circle_distance(a=None, b=None, iterable=False):
    geod = Geod(ellps='WGS84')

    if iterable:
        (a, b) = a

    distance = geod.inv(
        a.x, a.y,
        b.x, b.y,
    )[2]

    return distance

# Data

## Borders

In [ ]:
kaz = gpd.read_file('data/geoBoundaries-KAZ-ADM0_simplified.geojson')
ukr = gpd.read_file('data/geoBoundaries-UKR-ADM0_simplified.geojson')
rus = gpd.read_file('data/geoBoundaries-RUS-ADM0_simplified.geojson')
blr = gpd.read_file('data/geoBoundaries-BLR-ADM0_simplified.geojson')
borders = pd.concat((kaz, ukr, rus, blr)).reset_index(drop=True)
borders = borders.to_crs(epsg=4326)
borders

## Airports

In [ ]:
airports = pd.read_csv('data/airports.csv')
airports = gpd.GeoDataFrame(
    airports.drop(['GeoPointLat', 'GeoPointLong'], axis=1),
    geometry=gpd.points_from_xy(
        *airports[['GeoPointLong', 'GeoPointLat']].values.T
    ), crs='EPSG:4326'
)
airports[airports.ICAO.isin(['KJFK', 'UAAA'])]

# HEALPix

In [ ]:
NSIDE = 16
healpix = get_points(NSIDE)

NPIX = hp.nside2npix(NSIDE)
ipix = np.arange(NPIX)
depth = np.log2(NSIDE).astype(int)

print(NPIX, depth)

In [ ]:
lon, lat = healpix_to_lonlat(ipix, depth=depth, ellipsoid='WGS84')
lon = (lon + 180) % 360 - 180

In [ ]:
gdf = gpd.GeoDataFrame(
    healpix,
    geometry=gpd.points_from_xy(
        lon, lat
    ),
    crs='EPSG:4326',
)
gdf

In [ ]:
start = lonlat_to_healpix(
    airports[airports.ICAO=='UAAA'].geometry.x,
    airports[airports.ICAO=='UAAA'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]
end = lonlat_to_healpix(
    airports[airports.ICAO=='KJFK'].geometry.x,
    airports[airports.ICAO=='KJFK'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]

In [ ]:
df = pd.read_csv('point_pairs_full.csv', index_col=False)
df = df[~df.intersect_avoid].reset_index()
df

In [ ]:
df.drop(['index'], axis=1).to_csv('point_pairs_distances.csv', index=False)

# Graph

In [ ]:
G = nx.from_edgelist(
    df[['healpix_id_a', 'healpix_id_b']].values
)

In [ ]:
for idx, row in df.iterrows():
    G[row.healpix_id_a][row.healpix_id_b]['weight'] = row.distance

In [ ]:
route = nx.astar_path(
    G, start, end,
)

In [ ]:
gdf_route = gdf.iloc[route].geometry.reset_index()
gdf_route = gdf_route.rename({'index': 'point_id'}, axis=1)
gdf_route.point_id = gdf_route.point_id.astype(str)

gdf_route.loc[gdf_route.point_id==str(start), 'point_id'] = 'UAAA'
gdf_route.loc[gdf_route.point_id=='UAAA', 'geometry'] = airports.loc[airports.ICAO=='UAAA', 'geometry'].values

gdf_route.loc[gdf_route.point_id==str(end), 'point_id'] = 'KJFK'
gdf_route.loc[gdf_route.point_id=='KJFK', 'geometry'] = airports.loc[airports.ICAO=='KJFK', 'geometry'].values

gdf_route

In [ ]:
countries = ['KAZ', 'UKR', 'RUS', 'BLR', 'USA']

fig = go.Figure()

trace = go.Choropleth(
    locations=countries,
    z=list(range(len(countries))),
    locationmode='ISO-3',
    colorscale=np.array(px.colors.qualitative.Plotly)[[2, 1, 1, 1, 4]],
    text=countries,
    showscale=False,
)
fig.add_trace(trace)

route_trace = go.Scattergeo(
    lon=gdf_route.geometry.x,
    lat=gdf_route.geometry.y,
    mode='lines',
    marker=dict(size=1, color='blue'),
)
fig.add_trace(route_trace)

start_end = go.Scattergeo(
    lon=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.x,
    lat=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.y,
    mode='lines+markers',
    marker=dict(size=5, color='red'),
)
fig.add_trace(start_end)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
    geo=dict(
        showframe=True,
        showland=True,
    ),
)
fig.show()

In [ ]:
from pint import UnitRegistry
ureg = UnitRegistry()

In [ ]:
route_segments = pd.DataFrame([
    {
        'segment': f'{window.point_id.values[0]} - {window.point_id.values[1]}',
        'distance_m': great_circle_distance(*window.geometry.values),
        'distance_nml': (
            great_circle_distance(*window.geometry.values) * ureg.meter
        ).to(ureg.nautical_mile).magnitude,
    }
    for window in gdf_route.rolling(2)
    if len(window) == 2
])
route_segments

In [ ]:
print(
    f'direct UAAA-KJFK: {(
        great_circle_distance(
            *gdf_route[
                gdf_route.point_id.isin(['UAAA', 'KJFK'])
            ].geometry.values
        ) * ureg.meter
    ).to(ureg.nautical_mile).magnitude:.2f} nautical miles'
)
print(f'segmented UAAA-KJFK: {route_segments.distance_nml.sum():.2f} nautical miles')